# Step 6: Privacy Extensions
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## The Problem with Standard Federated Learning

Standard FL already protects raw patient data — images never leave the hospital.
But it still has a privacy gap: **the server sees each client's model weights**.

Why is this a risk?
> Researchers have shown that it's sometimes possible to **reconstruct training images** from model gradients alone. This is called a **"gradient inversion attack"** or **"model inversion attack."**

```
Attacker's goal: given weights w, find image x such that model(x) → label
```

## Two Solutions Covered in This Notebook

| Technique | What the server sees | Privacy level | Speed |
|-----------|---------------------|---------------|-------|
| Standard FL | Plaintext weights | Low | Fast |
| **Secure Aggregation** | Only the SUM of weights | Medium | Fast |
| **Homomorphic Encryption** | Only encrypted ciphertext | High | Slow |

Both techniques produce the **same aggregated model** as standard FL — no accuracy is lost!

## 🔧 Install TenSEAL (Homomorphic Encryption Library)

[TenSEAL](https://github.com/OpenMined/TenSEAL) is an open-source Python library for homomorphic encryption built on Microsoft SEAL.

It uses the **CKKS scheme**, which supports approximate arithmetic on floating-point numbers — perfect for averaging neural network weights.

**This may take 1–2 minutes to install.** Run this cell first.

In [ ]:
!pip install tenseal -q
print("Installation complete!")

## 🔧 Mount Google Drive

We need the client data files from notebook 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, copy, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

try:
    import tenseal as ts
    HE_AVAILABLE = True
    print("TenSEAL loaded successfully.")
except ImportError:
    HE_AVAILABLE = False
    print("TenSEAL not available — will show conceptual simulation instead.")

DRIVE_BASE  = "/content/drive/MyDrive/FederatedLearning"
DATA_DIR    = os.path.join(DRIVE_BASE, "data")
RESULTS_DIR = os.path.join(DRIVE_BASE, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

for fname in ["client1.npz", "client2.npz", "test_data.npz"]:
    status = "✅" if os.path.exists(os.path.join(DATA_DIR, fname)) else "❌ Missing — run notebook 2 first"
    print(f"  {fname}: {status}")

## Setup: Model, Helpers, and Data

In [ ]:
NUM_CLASSES   = 7
BATCH_SIZE    = 64
LEARNING_RATE = 0.001
LOCAL_EPOCHS  = 2
NUM_ROUNDS    = 3   # Fewer rounds — HE is slow

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


def make_loader(images, labels, shuffle=True):
    x = torch.tensor(images / 255.0, dtype=torch.float32).permute(0, 3, 1, 2)
    y = torch.tensor(labels, dtype=torch.long)
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
            total   += lbls.size(0)
    return correct / total


def client_local_train(global_model, loader, local_epochs, lr):
    local_model = copy.deepcopy(global_model)
    local_model.train()
    opt  = optim.Adam(local_model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(local_epochs):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            crit(local_model(imgs), lbls).backward()
            opt.step()
    return local_model.state_dict()


def weights_to_flat(state_dict):
    return np.concatenate([v.cpu().numpy().flatten() for v in state_dict.values()])


def flat_to_weights(flat, reference):
    out, offset = {}, 0
    for key, t in reference.items():
        n = t.numel()
        out[key] = torch.tensor(flat[offset:offset+n].reshape(t.shape), dtype=t.dtype)
        offset += n
    return out


# Load data
c1 = np.load(os.path.join(DATA_DIR, "client1.npz"))
c2 = np.load(os.path.join(DATA_DIR, "client2.npz"))
td = np.load(os.path.join(DATA_DIR, "test_data.npz"))
client1_loader = make_loader(c1["images"], c1["labels"])
client2_loader = make_loader(c2["images"], c2["labels"])
test_loader    = make_loader(td["images"], td["labels"], shuffle=False)
client_sizes   = [len(c1["labels"]), len(c2["labels"])]
total_samples  = sum(client_sizes)

print(f"\nData loaded. Client 1: {client_sizes[0]}, Client 2: {client_sizes[1]}")

---
# Part 1: Homomorphic Encryption

## What is Homomorphic Encryption?

Homomorphic Encryption (HE) allows a server to **perform computations on encrypted data** — without ever decrypting it.

The key mathematical property:

```
Enc(a) + Enc(b)  =  Enc(a + b)
Enc(a) × scalar  =  Enc(a × scalar)
```

Applied to FL aggregation:
```
Step 1 (Client 1): Encrypt weights  →  Enc(w1)
Step 2 (Client 2): Encrypt weights  →  Enc(w2)
Step 3 (Server)  : Enc(avg) = Enc(w1) × f1 + Enc(w2) × f2   ← server cannot read w1 or w2!
Step 4 (Any client): Decrypt Enc(avg) → avg = w1×f1 + w2×f2  ← identical to plaintext average
```

The server produces the **same averaged model** as standard FL, but **never sees any individual weight values**.

### About CKKS (the scheme we use)
- Designed for **floating-point arithmetic** (needed for neural network weights)
- **Approximate** — introduces tiny numerical error (< 1e-5), negligible for ML
- Based on the **Learning With Errors (LWE)** hardness assumption
- Widely used in privacy-preserving ML (Apple, Google, Microsoft)

In [ ]:
# -----------------------------------------------------------------------
# Quick HE concept demo: show encrypt → compute → decrypt on 5 numbers
# -----------------------------------------------------------------------
print("=" * 55)
print("  HE CONCEPT DEMO (small example)")
print("=" * 55)

if HE_AVAILABLE:
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
                     coeff_mod_bit_sizes=[60, 40, 40, 60])
    ctx.generate_galois_keys()
    ctx.global_scale = 2**40

    w1_example = [0.42, -0.17,  0.93,  0.05, -0.61]
    w2_example = [0.11,  0.33, -0.28,  0.77,  0.04]

    print(f"  Client 1 weights: {w1_example}")
    print(f"  Client 2 weights: {w2_example}")

    enc1 = ts.ckks_vector(ctx, w1_example)
    enc2 = ts.ckks_vector(ctx, w2_example)

    print(f"\n  After encryption the server sees ciphertext objects.")
    print(f"  The server CANNOT call .decrypt() — it does not have the secret key.")

    # Server performs weighted average on ciphertext
    f1, f2 = 0.5, 0.5  # equal weights for this simple example
    enc_avg = enc1 * f1 + enc2 * f2
    print(f"  Server computes: Enc(avg) = Enc(w1)×{f1} + Enc(w2)×{f2}  (still encrypted)")

    # Client decrypts
    result    = enc_avg.decrypt()
    expected  = [w1*f1 + w2*f2 for w1, w2 in zip(w1_example, w2_example)]
    max_error = max(abs(r - e) for r, e in zip(result, expected))

    print(f"\n  Decrypted average: {[round(r, 4) for r in result]}")
    print(f"  Plaintext average: {[round(e, 4) for e in expected]}")
    print(f"  Max numerical error: {max_error:.2e}  ← negligible (CKKS is approximate)")
    print(f"\n  ✅ HE produces the SAME result as plaintext averaging!")

else:
    w1_example = [0.42, -0.17,  0.93,  0.05, -0.61]
    w2_example = [0.11,  0.33, -0.28,  0.77,  0.04]
    expected   = [(a+b)/2 for a, b in zip(w1_example, w2_example)]

    print(f"  Client 1 weights  : {w1_example}")
    print(f"  Client 2 weights  : {w2_example}")
    print(f"  Server sees       : [ciphertext — unreadable numbers like 4729384, -8823619, ...]")
    print(f"  After decryption  : {[round(e, 4) for e in expected]}")
    print(f"  Plaintext average : {[round(e, 4) for e in expected]}  ← identical ✓")
    print(f"\n  Install TenSEAL to run the real demo: !pip install tenseal")

## HE Applied to FL: Aggregate Encrypted Model Weights

Now we apply HE to real model weights from the DermaMNIST CNN.
The model has ~500,000 parameters — we process them in chunks because CKKS has a slot limit.

**Expected timing on Colab CPU:**
- Encryption: ~15–30 seconds
- Encrypted aggregation: ~5–10 seconds
- Decryption: ~5–10 seconds

This is why HE is used selectively in practice — it is 10–100× slower than plaintext aggregation.

In [ ]:
print("Training clients locally (one round)...")
global_model_he = SimpleCNN(NUM_CLASSES).to(device)
w1_dict = client_local_train(global_model_he, client1_loader, LOCAL_EPOCHS, LEARNING_RATE)
w2_dict = client_local_train(global_model_he, client2_loader, LOCAL_EPOCHS, LEARNING_RATE)

w1_flat = weights_to_flat(w1_dict).astype(np.float64)
w2_flat = weights_to_flat(w2_dict).astype(np.float64)
n_params = len(w1_flat)
print(f"Total model parameters: {n_params:,}")

if HE_AVAILABLE:
    CHUNK = 4096  # Max CKKS slot size for poly_modulus_degree=8192

    # Set up CKKS context
    ctx = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
                     coeff_mod_bit_sizes=[60, 40, 40, 60])
    ctx.generate_galois_keys()
    ctx.global_scale = 2**40

    # ----- STEP 1: Clients encrypt their weights -----
    print(f"\n[Step 1] Clients encrypting weights ({n_params:,} params in chunks of {CHUNK})...")
    t0 = time.time()
    enc_w1 = [ts.ckks_vector(ctx, w1_flat[i:i+CHUNK].tolist()) for i in range(0, n_params, CHUNK)]
    enc_w2 = [ts.ckks_vector(ctx, w2_flat[i:i+CHUNK].tolist()) for i in range(0, n_params, CHUNK)]
    enc_time = time.time() - t0
    print(f"  Done in {enc_time:.1f}s  — {len(enc_w1)} chunks")

    # ----- STEP 2: Server aggregates encrypted weights (FedAvg) -----
    # The server has no secret key — it cannot read any individual weight!
    print(f"\n[Step 2] Server aggregating ENCRYPTED weights (FedAvg)...")
    f1 = client_sizes[0] / total_samples
    f2 = client_sizes[1] / total_samples
    t0 = time.time()
    enc_avg = [c1 * f1 + c2 * f2 for c1, c2 in zip(enc_w1, enc_w2)]
    agg_time = time.time() - t0
    print(f"  Done in {agg_time:.1f}s")

    # ----- STEP 3: Clients decrypt the aggregated result -----
    print(f"\n[Step 3] Client decrypting aggregated weights...")
    t0 = time.time()
    avg_he = np.concatenate([chunk.decrypt() for chunk in enc_avg])[:n_params]
    dec_time = time.time() - t0
    print(f"  Done in {dec_time:.1f}s")

    # Verify correctness
    avg_plain = w1_flat * f1 + w2_flat * f2
    max_err   = np.max(np.abs(avg_he - avg_plain))
    print(f"\n  Max error vs. plaintext average: {max_err:.2e}  (should be < 1e-3)")

    # Load into model and evaluate
    global_model_he.load_state_dict(flat_to_weights(avg_he.astype(np.float32), w1_dict))
    he_acc = evaluate(global_model_he, test_loader)
    print(f"  HE-FL accuracy after 1 round: {he_acc:.2%}")

    total_he_time = enc_time + agg_time + dec_time
    print(f"\n  Total HE overhead: {total_he_time:.1f}s  (vs. ~0s for plaintext averaging)")

else:
    print("\n[Skipped — TenSEAL not available]")
    print("Install with: !pip install tenseal  then restart the kernel")

---
# Part 2: Secure Aggregation with Additive Masking

## The Idea: Masks That Cancel Out

Secure Aggregation is a **cryptographic protocol** that lets the server compute the **sum** of client weights — but prevents the server from seeing any individual client's weights.

### How pairwise masking works (2 clients)

```
Before the round:
  Client 1 and Client 2 agree on a shared random mask r
  (using a Diffie-Hellman key exchange over a secure channel)

When sending weights:
  Client 1 sends:  w1 + r    (masked)
  Client 2 sends:  w2 - r    (masked with opposite sign)

Server adds them:
  (w1 + r) + (w2 - r) = w1 + w2   ← masks cancel!

Server divides to get the average:
  (w1 + w2) / 2 = correct average ✓
```

The server **cannot** recover w1 or w2 individually — it only sees two masked values that look like noise.

### Why is this better than just encrypting?
- **Much faster** than HE — no heavy encryption needed
- Works with **any aggregation function** (not just addition)
- The shared mask is never sent to the server — it's generated locally from a shared seed

### Real-world use
Google uses Secure Aggregation in Gboard (keyboard app) FL since 2017.

In [ ]:
print("=" * 55)
print("  SECURE AGGREGATION CONCEPT DEMO")
print("=" * 55)

# Small example first — 5 weights
np.random.seed(7)
w1 = np.array([0.42, -0.17,  0.93,  0.05, -0.61])
w2 = np.array([0.11,  0.33, -0.28,  0.77,  0.04])
true_avg = (w1 + w2) / 2  # what the server should compute

# Clients agree on a shared random mask (via secure channel)
shared_seed = 12345
rng  = np.random.RandomState(shared_seed)
mask = rng.randn(5)  # both clients can generate this from the shared seed

# What each client sends
masked_w1 = w1 + mask
masked_w2 = w2 - mask

print("  True Client 1 weights: ", w1.round(3).tolist())
print("  True Client 2 weights: ", w2.round(3).tolist())
print()
print(f"  Shared mask (secret):  ", mask.round(3).tolist())
print()
print("  What Client 1 sends:   ", masked_w1.round(3).tolist(), "  ← looks like noise")
print("  What Client 2 sends:   ", masked_w2.round(3).tolist(), "  ← looks like noise")
print()

# Server adds the two masked values
server_sum = masked_w1 + masked_w2
server_avg = server_sum / 2

print("  Server computes sum:   ", server_sum.round(3).tolist())
print("  Server computes avg:   ", server_avg.round(3).tolist())
print("  True average:          ", true_avg.round(3).tolist())
print()
print(f"  Max error: {np.max(np.abs(server_avg - true_avg)):.2e}  ← masks cancelled perfectly")
print()
print("  KEY INSIGHT: The server sees only masked_w1 and masked_w2.")
print("  Without knowing the mask, it CANNOT recover w1 or w2.")

## Secure Aggregation Applied to Full FL Training

Now we run multiple FL rounds using additive masking — and compare to standard FL.

The accuracy should be **identical** (or nearly so) to standard FL, because the masks cancel out perfectly.

In [ ]:
print(f"Running {NUM_ROUNDS} FL rounds: Standard vs. Secure Aggregation\n")

# Initialize both models with the same weights for a fair comparison
global_std = SimpleCNN(NUM_CLASSES).to(device)
global_sa  = SimpleCNN(NUM_CLASSES).to(device)
global_sa.load_state_dict(copy.deepcopy(global_std.state_dict()))

hist_std, hist_sa = [], []

for rnd in range(1, NUM_ROUNDS + 1):
    print(f"  ===== Round {rnd}/{NUM_ROUNDS} =====")

    # === Standard FL ===
    w1s = client_local_train(global_std, client1_loader, LOCAL_EPOCHS, LEARNING_RATE)
    w2s = client_local_train(global_std, client2_loader, LOCAL_EPOCHS, LEARNING_RATE)
    f1  = client_sizes[0] / total_samples
    f2  = client_sizes[1] / total_samples
    w1f_std = weights_to_flat(w1s)
    w2f_std = weights_to_flat(w2s)
    avg_std = w1f_std * f1 + w2f_std * f2
    global_std.load_state_dict(flat_to_weights(avg_std, w1s))

    # === Secure Aggregation (additive masking) ===
    w1a = client_local_train(global_sa, client1_loader, LOCAL_EPOCHS, LEARNING_RATE)
    w2a = client_local_train(global_sa, client2_loader, LOCAL_EPOCHS, LEARNING_RATE)
    w1f_sa = weights_to_flat(w1a)
    w2f_sa = weights_to_flat(w2a)
    n = len(w1f_sa)

    # Clients generate mask from a shared random seed
    shared_seed = np.random.randint(0, 2**31)
    mask = np.random.RandomState(shared_seed).randn(n).astype(np.float32) * 0.01

    masked_w1 = w1f_sa + mask   # Client 1 adds mask
    masked_w2 = w2f_sa - mask   # Client 2 subtracts mask

    # Server sums masked weights — mask residual should be ~0
    server_avg_sa = masked_w1 * f1 + masked_w2 * f2
    global_sa.load_state_dict(flat_to_weights(server_avg_sa, w1a))

    # What the server observed vs. the true weights:
    print(f"  [Server sees masked values — sample weight[0]:]")
    print(f"    Client 1 true: {w1f_sa[0]:+.5f}   sent: {masked_w1[0]:+.5f}")
    print(f"    Client 2 true: {w2f_sa[0]:+.5f}   sent: {masked_w2[0]:+.5f}")

    # Evaluate both models
    acc_std = evaluate(global_std, test_loader)
    acc_sa  = evaluate(global_sa,  test_loader)
    hist_std.append(acc_std)
    hist_sa.append(acc_sa)

    print(f"  Standard FL accuracy     : {acc_std:.2%}")
    print(f"  Secure Aggregation accuracy: {acc_sa:.2%}\n")

print(f"Final — Standard FL: {hist_std[-1]:.2%} | Secure Aggregation: {hist_sa[-1]:.2%}")
print(f"Accuracy difference: {hist_sa[-1] - hist_std[-1]:+.4f}  (should be near 0)")

---
# Part 3: Comparison & Visualization

In [ ]:
rounds = list(range(1, NUM_ROUNDS + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Privacy-Preserving FL: Standard vs. Secure Aggregation", fontsize=13)

# Left: accuracy over rounds
ax = axes[0]
ax.plot(rounds, hist_std, marker="o", color="steelblue",  label="Standard FL",        linewidth=2)
ax.plot(rounds, hist_sa,  marker="s", color="seagreen",   label="Secure Aggregation", linewidth=2, linestyle="--")
ax.set_xlabel("Round")
ax.set_ylabel("Test Accuracy")
ax.set_title("Accuracy per Round")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: privacy techniques comparison radar-style bar chart
ax = axes[1]
techniques  = ["Standard FL", "Secure Agg.", "HE-FL"]
privacy_lvl = [1, 3, 5]          # 1=low, 5=high (conceptual scale)
speed_lvl   = [5, 5, 1]          # 1=slow, 5=fast
x = np.arange(len(techniques))
w = 0.35
ax.bar(x - w/2, privacy_lvl, width=w, label="Privacy (↑ better)", color="tomato",    alpha=0.8)
ax.bar(x + w/2, speed_lvl,   width=w, label="Speed (↑ faster)",   color="steelblue", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(techniques)
ax.set_ylim(0, 6)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["Very Low", "Low", "Medium", "High", "Very High"])
ax.set_title("Privacy vs. Speed Tradeoff\n(conceptual scale)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(RESULTS_DIR, "plots", "privacy_comparison.png")
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
plt.savefig(plot_path, dpi=120)
print(f"Saved: {plot_path}")
plt.show()

## Full Comparison Table

| Technique | What server sees | Accuracy loss | Compute cost | Used in production? |
|-----------|-----------------|--------------|-------------|--------------------|
| **Standard FL** | Plaintext weights | None | Low | Yes (most systems) |
| **Secure Aggregation** | Only the SUM | None | Low | Yes (Google Gboard) |
| **Homomorphic Encryption** | Encrypted ciphertext | None (tiny approx. error) | High (10–100×) | Research / limited prod. |
| **Differential Privacy** | Noisy weights | Small (tunable) | Low | Yes (Apple, Google) |

---

## Key Takeaways

1. **Secure Aggregation** is the most practical immediate upgrade to FL privacy — no accuracy cost, little speed cost.

2. **Homomorphic Encryption** gives the strongest guarantees but is expensive. Active research is making it faster every year.

3. **The threat model matters.** Are you protecting against:
   - A curious server? → Secure Aggregation is enough
   - An adversary who sees all messages? → Add Differential Privacy
   - A fully malicious server? → HE + secure protocols

4. **No single technique is a silver bullet.** Real privacy-preserving ML systems combine multiple layers of protection.

---

## Further Reading
- Bonawitz et al. (2017) — *Practical Secure Aggregation for Privacy-Preserving Machine Learning*
- Chiang et al. (2021) — *Gradient Inversion Attacks and Defenses in FL*
- TenSEAL library: https://github.com/OpenMined/TenSEAL

In [ ]:
# Save results
results = {
    "secure_aggregation_acc": [round(a, 4) for a in hist_sa],
    "standard_fl_acc":        [round(a, 4) for a in hist_std],
    "he_available":            HE_AVAILABLE,
}
with open(os.path.join(RESULTS_DIR, "privacy_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to Google Drive.")
print("\n✅ Lab complete! You have now built a full federated learning system")
print("   with privacy-preserving aggregation techniques.")